Install packages

In [21]:
# !module load py-mpi4py
# !pip install mpi4py

In [22]:
!pip install fastparquet mplsoccer

Load packages

In [ ]:
import time
import sys, os
import pandas as pd
import numpy as np
# import matplotlib.pyplot as plt
# from mpi4py import MPI

In [24]:
sys.path.append("xthreat-research-v1")

In [25]:
from xThreat import xThreat

Specify the parameters for the sampling

In [ ]:
# Specify the sample sizes, the grids and the number of bootstraps
sample_sizes = [
    4000000, 1_300_000,  630_000,  370_000,  240_000,  170_000,  130_000,  100_000,
    ]
grid_sizes = [
    (16, 12), (32, 24), (40, 30), (48, 36), (56, 42), (64, 48), # The last one takes a large amount of time
              ]
n_bootstraps = 1000 # Total number of bootstraps
bootstrap_range = 500 # The bootstraps performed in this notebook

Import the data

In [ ]:
data_path = 'xThreat_data_v2.parquet'
df_events = pd.read_parquet(data_path, engine='fastparquet')

df_events['shot'] = ~df_events['shot_outcome'].isna()
df_events['goal'] = df_events['shot_outcome'] == 'Goal'

df_events.drop(columns=['id', 'shot_outcome', 'possession'], inplace=True)
df_size_bytes = df_events.memory_usage(deep=True).sum()
df_size_mb = df_size_bytes / (1024 ** 2)  # Convert to MB
print(f"DataFrame size: {df_size_mb:.2f} MB")

DataFrame size: 1009.50 MB


In [27]:
# Train an xThreat model and apply _filter_out_of_bounds.
# In this way, the other models won't have to do that during fit.
# Will speed up computations.
xThreat_prefit = xThreat(16, 12)
xThreat_prefit.fit(df_events)
df_events = xThreat_prefit._filter_out_of_bounds(df_events)

Filtered out 3 events that had coordinates out of bounds.
In this, there were 0 shots and 0 goals.
Filtered out 3 events that had coordinates out of bounds.
In this, there were 0 shots and 0 goals.


Assign a random_state to each situation.

In [29]:
# Create a list with all combinations and assign a unique random_state
sampling_params = {}
for n_x, n_y in grid_sizes:
    for sample_size in sample_sizes:
        for i_bootstrap in range(n_bootstraps):
            random_state = 42 + (n_x + n_y) * n_bootstraps * 42 + sample_size
            sampling_params[(sample_size, n_x, n_y, i_bootstrap)] = random_state
print(len(sampling_params))

960


In [30]:
cell_begin_time = time.time()

max_size_store_trans_matr = 0
sum_size_store_trans_matr = 0
n = 0

sampled_params = {}

for n_x, n_y in grid_sizes:
    begin_time = time.time()
    
    # Fit the 'true' xThreat model
    xT_true = xThreat(n_x, n_y)
    xT_true.fit(df_events, filter_events=False, convergence_threshold=1e-9)

    for sample_size in sample_sizes:
        for i_bootstrap in range(n_bootstraps):
            
            # Sample from the true model
            random_state = sampling_params[(sample_size, n_x, n_y, i_bootstrap)]
            df_sample = xT_true.sample(sample_size, random_state=random_state)

            # Fit a model on the sample
            xT_resampled = xThreat(n_x, n_y)
            xT_resampled.fit(df_sample)

            # Save the resampled method
            xT_resampled.save_to_pickle(f'model-storage/xt-N{sample_size}-n_x{n_x}-n_y{n_y}-i_bootstrap{i_bootstrap}-random_state{random_state}.pickle' )

            # Keep track of the storage size of the models
            max_size_store_trans_matr = max(max_size_store_trans_matr, sys.getsizeof(xT_resampled.transition_matrix))
            n += 1
            sum_size_store_trans_matr += sys.getsizeof(xT_resampled.transition_matrix)
    
    # Print the time it took to do the samples for this grid size
    hours, remainder = divmod(time.time()-begin_time, 3600)
    minutes, seconds = divmod(remainder, 60)
    print(f'Performed the {n_bootstraps} bootstrap with grid {(n_x, n_y)} within {int(hours)}h, {int(minutes)}m, {int(seconds)}s.\n')

# Print the time it took to do the sampling
hours, remainder = divmod(time.time()-cell_begin_time, 3600)
minutes, seconds = divmod(remainder, 60)
print(f'The whole resampling process took {int(hours)}h, {int(minutes)}m, {int(seconds)}s')
print(f'The maximal size of the transition matrix was {max_size_store_trans_matr/1024**2:.2f}MB')
print(f'The average size of the transition matrix was {sum_size_store_trans_matr/n/1024**2:.2f}MB')


Performed the 20 bootstrap with grid (16, 12) within 0h, 0m, 40s.

Performed the 20 bootstrap with grid (32, 24) within 0h, 0m, 45s.

Performed the 20 bootstrap with grid (40, 30) within 0h, 0m, 51s.

Performed the 20 bootstrap with grid (48, 36) within 0h, 0m, 59s.

Performed the 20 bootstrap with grid (56, 42) within 0h, 1m, 20s.

Performed the 20 bootstrap with grid (64, 48) within 0h, 1m, 36s.

The whole resampling process took 0h, 6m, 14s
The maximal size of the transition matrix was 72.09MB
The average size of the transition matrix was 25.51MB


In [17]:
3*60/4

45.0

In [ ]:
## 10 took 3:38min

In [31]:
1000/20

50.0

In [33]:
50*6.5/60

5.416666666666667